# UFO datetime fix

# IMPORTS

In [1]:
from pathlib import Path
import pandas as pd
import geopandas as gpd
import os, sys

import matplotlib.pyplot as plt
import matplotlib
import numpy as np

import re
import rasterio
from shapely.geometry import box
from rasterio.warp import transform_bounds
 
 

In [2]:
import ipywidgets
from tqdm import tqdm

# PARAMETERS

In [3]:

OUT_DIR = Path('/home/cefect/LS/10_IO/UFO')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# LOAD DATA

## [NO: these are broken] STAC files

downloaded from https://zenodo.org/records/15238470

In [4]:
# search_fp = Path('/home/cefect/LS/10_IO/UFO/inputs/labels')
# assert search_fp.exists(), f"Input folder {search_fp} does not exist"

scan the directory and concat all the stac jsons

In [5]:


# records = []
# for dirpath, dirs, files in tqdm(os.walk(search_fp), desc="Processing directories"):
#     for f in files:
#         if f.endswith(".json"):
#             fp = os.path.join(dirpath, f)
#             try:
#                 gdf = gpd.read_file(fp)
#                 records.append(gdf)
#             except Exception:
#                 pass

# concat_gdf = gpd.pd.concat(records, ignore_index=True)
# concat_gdf = concat_gdf.set_crs(4326)
# concat_gdf.info()

## Extract Geometry from Tiles

In [6]:
search_dir = Path('/home/cefect/LS/10_IO/UFO/inputs/labels_tif')
assert search_dir.exists(), f"Input folder {search_dir} does not exist"

In [7]:
tif_files = list(search_dir.glob('*.tif'))
print(f"Found {len(tif_files)} .tif files in {search_dir}")

Found 231 .tif files in /home/cefect/LS/10_IO/UFO/inputs/labels_tif


In [8]:

extents = []
crs_set = set()
for tif_fp in tif_files:
    with rasterio.open(tif_fp) as src:
        if src.crs is None:
            print(f"Skipping {tif_fp.name}: missing CRS")
            continue

        bounds = src.bounds
        try:
            bounds_4326 = transform_bounds(
                src.crs,
                'EPSG:4326',
                bounds.left,
                bounds.bottom,
                bounds.right,
                bounds.top,
                densify_pts=21,
            )
        except Exception as e:
            print(f"Skipping {tif_fp.name}: failed to transform bounds from {src.crs}: {e}")
            continue

        left, bottom, right, top = bounds_4326
        if not (-180 <= left <= 180 and -180 <= right <= 180 and -90 <= bottom <= 90 and -90 <= top <= 90):
            print(f"Skipping {tif_fp.name}: bounds outside WGS84 range: {bounds_4326}")
            continue

        crs_set.add(str(src.crs))
        extents.append({
            'file': tif_fp.name,
            'left': left,
            'bottom': bottom,
            'right': right,
            'top': top,
        })

print(f"extracted {len(extents)} valid extents from {len(tif_files)} rasters (CRS seen={sorted(crs_set)})")
pd.DataFrame(extents).to_csv('UFO_tif_extents.csv', index=False)



extracted 231 valid extents from 231 rasters (CRS seen=['EPSG:32615', 'EPSG:32616', 'EPSG:4326'])


In [9]:


# Create a GeoDataFrame from extents
gdf_extents = gpd.GeoDataFrame(
    extents,
    geometry=[box(e['left'], e['bottom'], e['right'], e['top']) for e in extents],
    crs='EPSG:4326'
).loc[:, ['file', 'geometry']]
gdf_extents.head()

,file,geometry
0,HTX_7hhv.tif,"POLYGON ((-95.63073 29.94307, -95.63073 29.971..."
1,HTX_5h5l.tif,"POLYGON ((-95.24615 28.97712, -95.24615 29.005..."
2,DKA_ond8.tif,"POLYGON ((90.36849 23.92214, 90.36849 23.95185..."
3,HTX_me9z.tif,"POLYGON ((-95.48493 29.4443, -95.48493 29.4725..."
4,CTO_ebrl.tif,"POLYGON ((105.82563 10.01024, 105.82563 10.038..."


extract fields from the filename

In [10]:
concat_gdf = (
    gdf_extents
    .assign(
        UFO_event_id=lambda df: df['file'].str.split('_').str[0],
        ps_id=lambda df: df['file'].str.split('_').str[1].str.replace('.tif', ''),
        Chip_id=lambda df: df['file'].str.replace('.tif', ''),
    )
)
concat_gdf = concat_gdf.loc[:, ['file', 'UFO_event_id', 'ps_id', 'Chip_id', 'geometry']]
concat_gdf.head()

,file,UFO_event_id,ps_id,Chip_id,geometry
0,HTX_7hhv.tif,HTX,7hhv,HTX_7hhv,"POLYGON ((-95.63073 29.94307, -95.63073 29.971..."
1,HTX_5h5l.tif,HTX,5h5l,HTX_5h5l,"POLYGON ((-95.24615 28.97712, -95.24615 29.005..."
2,DKA_ond8.tif,DKA,ond8,DKA_ond8,"POLYGON ((90.36849 23.92214, 90.36849 23.95185..."
3,HTX_me9z.tif,HTX,me9z,HTX_me9z,"POLYGON ((-95.48493 29.4443, -95.48493 29.4725..."
4,CTO_ebrl.tif,CTO,ebrl,CTO_ebrl,"POLYGON ((105.82563 10.01024, 105.82563 10.038..."


In [11]:
concat_gdf.to_file(OUT_DIR / 'UFO_labels_from_tif_extents.geojson', driver='GeoJSON')

light post to get columns for join

In [12]:
concat_gdf.drop(columns=['geometry'])

,file,UFO_event_id,ps_id,Chip_id
0,HTX_7hhv.tif,HTX,7hhv,HTX_7hhv
1,HTX_5h5l.tif,HTX,5h5l,HTX_5h5l
2,DKA_ond8.tif,DKA,ond8,DKA_ond8
3,HTX_me9z.tif,HTX,me9z,HTX_me9z
4,CTO_ebrl.tif,CTO,ebrl,CTO_ebrl
...,...,...,...,...
226,HTX_8zqe.tif,HTX,8zqe,HTX_8zqe
227,KTM_3AOH.tif,KTM,3AOH,KTM_3AOH
228,HTX_1g47.tif,HTX,1g47,HTX_1g47
229,CMO_bebg.tif,CMO,bebg,CMO_bebg


## Scene metadata

gsheet from Rohit

In [13]:
fp = Path('./PSnames.csv')
assert fp.exists(), f"File {fp} does not exist"

In [14]:
meta_df_raw = pd.read_csv(fp)
meta_df_raw.head()


,full_filename,short_filename
0,10_20190326_072743_1021_1ynv.tif,1ynv.tif
1,10_20190326_072743_1021_a69z.tif,a69z.tif
2,06_20190321_155335_0f2b_17rl.tif,17rl.tif
3,06_20190321_155335_0f2b_f51l.tif,f51l.tif
4,06_20190321_155335_0f2b_go0s.tif,go0s.tif


In [15]:
#extract info from filename
df = meta_df_raw
meta_df = (
    meta_df_raw
    .assign(
        ps_id=lambda df: df['short_filename'].str.replace('.tif', '', regex=False),
        date=lambda df: pd.to_datetime(
            df['full_filename'].str.split('_').str[1],
            format='%Y%m%d', errors='coerce'),
    )
)
print(meta_df.info())
meta_df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 378 entries, 0 to 377
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   full_filename   378 non-null    object        
 1   short_filename  378 non-null    object        
 2   ps_id           378 non-null    object        
 3   date            378 non-null    datetime64[ns]
dtypes: datetime64[ns](1), object(3)
memory usage: 11.9+ KB
None


,full_filename,short_filename,ps_id,date
0,10_20190326_072743_1021_1ynv.tif,1ynv.tif,1ynv,2019-03-26
1,10_20190326_072743_1021_a69z.tif,a69z.tif,a69z,2019-03-26
2,06_20190321_155335_0f2b_17rl.tif,17rl.tif,17rl,2019-03-21
3,06_20190321_155335_0f2b_f51l.tif,f51l.tif,f51l,2019-03-21
4,06_20190321_155335_0f2b_go0s.tif,go0s.tif,go0s,2019-03-21


In [16]:
#investigate null dates
bx = meta_df['date'].isnull()
if bx.any():
    meta_df[bx]

# JOIN

In [17]:
meta_set = set(meta_df['ps_id'].unique())
concat_set = set(concat_gdf['ps_id'].unique())
missing_in_meta = concat_set - meta_set
missing_in_concat = meta_set - concat_set
print(f"Number of ps_id in concat_gdf not in meta_df: {len(missing_in_meta)}")
print(f"Number of ps_id in meta_df not in concat_gdf: {len(missing_in_concat)}")

Number of ps_id in concat_gdf not in meta_df: 0
Number of ps_id in meta_df not in concat_gdf: 147


In [18]:

join_df = (
    concat_gdf.set_index('ps_id', verify_integrity=True)
    .join(
        meta_df.set_index('ps_id', verify_integrity=True)['date'].rename('datetime_fixed'),
        how='left',
    )
)
print(join_df.info())
join_df.head()

<class 'geopandas.geodataframe.GeoDataFrame'>
Index: 231 entries, 7hhv to 2p5c
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   file            231 non-null    object        
 1   UFO_event_id    231 non-null    object        
 2   Chip_id         231 non-null    object        
 3   geometry        231 non-null    geometry      
 4   datetime_fixed  231 non-null    datetime64[ns]
dtypes: datetime64[ns](1), geometry(1), object(3)
memory usage: 10.8+ KB
None


,file,UFO_event_id,Chip_id,geometry,datetime_fixed
ps_id,,,,,
7hhv,HTX_7hhv.tif,HTX,HTX_7hhv,"POLYGON ((-95.63073 29.94307, -95.63073 29.971...",2017-08-31
5h5l,HTX_5h5l.tif,HTX,HTX_5h5l,"POLYGON ((-95.24615 28.97712, -95.24615 29.005...",2017-08-31
ond8,DKA_ond8.tif,DKA,DKA_ond8,"POLYGON ((90.36849 23.92214, 90.36849 23.95185...",2020-08-25
me9z,HTX_me9z.tif,HTX,HTX_me9z,"POLYGON ((-95.48493 29.4443, -95.48493 29.4725...",2017-08-31
ebrl,CTO_ebrl.tif,CTO,CTO_ebrl,"POLYGON ((105.82563 10.01024, 105.82563 10.038...",2019-10-02


## write

In [21]:
ofp = 'UFO_events_analysis_ready.geojson'
gdf = (
    join_df
    .reset_index(drop=False)
    .loc[:, ['ps_id', 'UFO_event_id', 'Chip_id', 'datetime_fixed', 'geometry']]
    .rename(columns={'datetime_fixed': 'datetime'})
    .sort_values(['UFO_event_id', 'ps_id'])
)

gpd.GeoDataFrame(gdf, crs='4326').to_file(ofp, driver='GeoJSON')
print(f"Wrote output to {ofp}")

Wrote output to UFO_events_analysis_ready.geojson
